# TensorFlow and Keras Basics

**Learning Objectives:** Master TensorFlow/Keras Sequential API for building, training, evaluating, and deploying neural networks. Apply to MNIST digit classification.

## 1. What is TensorFlow/Keras?

**TensorFlow** is an open-source ML platform by Google. **Keras** is TensorFlow's high-level API for building neural networks.

Key advantages:
- **Sequential API** — simple layer-by-layer stacking
- **Functional API** — complex topologies (multi-input, multi-output, shared layers)
- **Eager execution** — run operations immediately (no graph compilation needed)
- **Built-in callbacks** — early stopping, model checkpointing, learning rate scheduling
- **TensorBoard** — visualization of training

In [ ]:
# Install and import TensorFlow
# !pip install tensorflow

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

## 2. Building a Model with Sequential API

The **Sequential API** stacks layers linearly:

In [ ]:
# Build a simple feedforward NN
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,), name='hidden_1'),
    layers.Dense(64, activation='relu', name='hidden_2'),
    layers.Dense(10, activation='softmax', name='output')
], name='simple_mnist_model')

model.summary()

### Key Layer Parameters

| Parameter | Description | Example |
|-----------|-------------|---------|
| `units` | Number of neurons | `128` |
| `activation` | Activation function | `'relu'`, `'sigmoid'`, `'softmax'` |
| `input_shape` | Shape of input data | `(784,)` for flattened 28×28 |
| `kernel_regularizer` | Weight regularization | `l1()`, `l2()` |
| `name` | Layer name | `'hidden_1'` |

## 3. Compiling the Model

Compiling configures the learning process:
- **Optimizer** — algorithm for gradient descent (e.g., `'adam'`, `'sgd'`)
- **Loss** — objective to minimize (`'categorical_crossentropy'`, `'mse'`)
- **Metrics** — evaluation metrics to track (`'accuracy'`)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## 4. Loading and Preprocessing MNIST

MNIST: 70,000 grayscale 28×28 images of handwritten digits (0-9).

In [ ]:
# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Explore shapes
print(f"Training data: {x_train.shape}, labels: {y_train.shape}")
print(f"Test data: {x_test.shape}, labels: {y_test.shape}")

# Normalize pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Flatten 28x28 images to 784-dimensional vectors
x_train_flat = x_train.reshape(-1, 784)
x_test_flat = x_test.reshape(-1, 784)

# One-hot encode labels
y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)

print(f"After preprocessing: {x_train_flat.shape}, labels: {y_train_cat.shape}")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap='gray')
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Training with `fit()`

The `fit()` method trains the model with:
- **batch_size** — samples per gradient update
- **epochs** — full passes over the data
- **validation_split** — hold-out fraction for validation
- **callbacks** — hooks for early stopping, checkpointing, etc.
- **verbose** — output detail (0=silent, 1=bar, 2=epoch)

In [ ]:
history = model.fit(
    x_train_flat, y_train_cat,
    batch_size=32,
    epochs=10,
    validation_split=0.2,
    verbose=1
)

## 6. Visualizing Training History

The `history.history` dictionary contains per-epoch metrics.

In [ ]:
def plot_training_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    # Loss plot
    ax1.plot(history.history['loss'], label='Training Loss')
    ax1.plot(history.history['val_loss'], label='Validation Loss')
    ax1.set_title('Model Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(alpha=0.3)

    # Accuracy plot
    ax2.plot(history.history['accuracy'], label='Training Accuracy')
    ax2.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax2.set_title('Model Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_training_history(history)

## 7. Evaluation with `evaluate()`

`evaluate()` returns the loss and metrics on test data.

In [ ]:
test_loss, test_acc = model.evaluate(x_test_flat, y_test_cat, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

## 8. Prediction with `predict()`

`predict()` returns raw output (probabilities for softmax).

In [ ]:
# Predict on test samples
predictions = model.predict(x_test_flat[:10])
predicted_classes = np.argmax(predictions, axis=1)

print("True labels:", y_test[:10])
print("Predictions:", predicted_classes)
print("Confidence:", np.max(predictions, axis=1).round(3))

In [ ]:
# Visualize some predictions
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_test[i], cmap='gray')
    true = y_test[i]
    pred = predicted_classes[i]
    color = 'green' if true == pred else 'red'
    ax.set_title(f"True: {true}, Pred: {pred}", color=color)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 9. Saving and Loading Models

Keras supports two save formats:
- **SavedModel** (default, TF-native) — directory with assets
- **HDF5** (`.h5`) — single file, requires `h5py`

In [ ]:
# Save the model
model.save('mnist_model.keras')
print("Model saved as 'mnist_model.keras'")

# Load the model back
loaded_model = keras.models.load_model('mnist_model.keras')

# Verify loaded model works
loaded_pred = loaded_model.predict(x_test_flat[:5])
print(f"Predictions match: {np.allclose(predictions[:5], loaded_pred)}")

## 10. Using the Functional API

The **Functional API** handles non-linear topologies (multi-input, multi-output, skip connections):

In [ ]:
# Functional API example
inputs = keras.Input(shape=(784,), name='input_layer')
x = layers.Dense(128, activation='relu', name='hidden_1')(inputs)
x = layers.Dense(64, activation='relu', name='hidden_2')(x)
outputs = layers.Dense(10, activation='softmax', name='output')(x)

functional_model = keras.Model(inputs=inputs, outputs=outputs, name='functional_mnist')
functional_model.summary()

## 11. Callbacks

Callbacks provide hooks during training:

In [ ]:
# Train with callbacks
cb_model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,)),
    layers.Dense(10, activation='softmax')
])
cb_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
    keras.callbacks.ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True)
]

history_cb = cb_model.fit(
    x_train_flat, y_train_cat,
    batch_size=32, epochs=20,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

## 12. Interview Problems

### Problem 1: Build a Feedforward NN for MNIST

Build a model with 3 hidden layers (256, 128, 64 neurons, ReLU), output with softmax, train for 15 epochs with Adam, and achieve >98% test accuracy. Plot the training history.

### Problem 2: Visualize Training History

Create a function that plots training & validation loss/accuracy with proper labels, grid, and legend. Highlight the best epoch.

### Problem 3: Implement Custom Callback

Write a custom Keras callback that logs learning rate and gradients norm at the end of each batch.

In [ ]:
# Solution: Problem 1 — Deep model for MNIST
deep_model = keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

deep_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

hist = deep_model.fit(
    x_train_flat, y_train_cat,
    batch_size=64, epochs=15,
    validation_split=0.2, verbose=0
)

test_acc = deep_model.evaluate(x_test_flat, y_test_cat, verbose=0)[1]
print(f"Test accuracy: {test_acc:.4f}")
plot_training_history(hist)

In [ ]:
# Solution: Problem 3 — Custom callback for learning rate & gradient norm
class GradientLogger(keras.callbacks.Callback):
    def on_train_batch_end(self, batch, logs=None):
        if batch % 100 == 0:  # log every 100 batches
            lr = self.model.optimizer.lr.numpy()
            # Compute gradient norm from first layer
            weights = self.model.layers[0].kernel
            with tf.GradientTape() as tape:
                tape.watch(weights)
            # note: full gradient logging requires tape in train_step
            print(f"Batch {batch}: LR = {lr:.6f}")

cb_model.fit(
    x_train_flat[:10000], y_train_cat[:10000],
    batch_size=32, epochs=2,
    callbacks=[GradientLogger()],
    verbose=0
)

## Summary

- Keras **Sequential API** provides a quick way to stack layers linearly
- **`compile()`** configures optimizer, loss, and metrics
- **`fit()`** trains the model with rich options (batch size, epochs, validation, callbacks)
- **`evaluate()`** returns loss/metrics on test data
- **`predict()`** generates predictions on new data
- **Saving/loading** models enables deployment and reuse
- **Callbacks** (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint) improve training
- The **Functional API** supports complex architectures beyond sequential stacks